# Evaluation of DIME segment lengths

This notebook compares segment-level DIME scores obtained with different window
lengths and overlap configurations.

It expects one segment-score table per configuration and a table containing the
external D-values. All results are displayed directly as tables.

The notebook provides:

1. a compact comparison of the principal evaluation metrics;
2. a comprehensive diagnostics table;
3. descriptive corpus and analysis-sample statistics;
4. pairwise agreement statistics for all window configurations.


In [1]:
from pathlib import Path
from time import perf_counter
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple
import warnings

import numpy as np
import pandas as pd

from scipy.stats import mannwhitneyu, pearsonr, spearmanr
from sklearn.metrics import roc_auc_score
import statsmodels.formula.api as smf


## 2. Configuration

Adapt the input paths in the following cell. Dictionary order determines the column
order in the result tables.

`BASELINE_CONFIGURATION` identifies the window configuration used as the reference
for baseline agreement and numerical baseline differences.

Each segment table must contain:

- `speaker`
- `recording_condition`
- `combined_score`
- `samples_begin`

Each speaker-by-condition combination is treated as one continuous recording.
`SPEAKER_CONDITION_AGGREGATION` determines how segment scores are aggregated for
comparisons with the external D-values and with other window configurations.


In [ ]:
# ---------------------------------------------------------------------
# Input paths
# ---------------------------------------------------------------------

SEGMENT_SCORE_PATHS = {
    "10s": Path(
        r"...\TS_10s\dialectality_segment_scores.csv"
    ),
    "9s": Path(
        r"...S\TS_9s\dialectality_segment_scores.csv"
    ),
    "8s": Path(
        r"...\TS_8s\dialectality_segment_scores.csv"
    ),
    "8s (4s ov.)": Path(
        r"...\TS_8s_4s\dialectality_segment_scores.csv"
    ),
    "5s": Path(
        r"...\TS_5s\dialectality_segment_scores.csv"
    ),
    "3s": Path(
        r"...\TS_3s\dialectality_segment_scores.csv"
    ),
    "1s": Path(
        r"...\TS_1s\dialectality_segment_scores.csv"
    ),
}

D_VALUES_PATH = Path("data/d_values.csv")

BASELINE_CONFIGURATION = "10s"

# ---------------------------------------------------------------------
# Duration metadata used by the descriptive statistics block
# ---------------------------------------------------------------------

# Verify this value if the segment-start indices were generated at a
# different sampling rate. `samples_begin` is interpreted in samples.
SAMPLE_RATE_HZ = 16_000

# Effective window duration for every configured segment table.
# The overlap configuration still uses an 8-second window; overlap is
# handled when unique analyzed audio duration is calculated.
WINDOW_LENGTH_SECONDS = {
    "10s": 10.0,
    "9s": 9.0,
    "8s": 8.0,
    "8s (4s ov.)": 8.0,
    "5s": 5.0,
    "3s": 3.0,
    "1s": 1.0,
}

# ---------------------------------------------------------------------
# Input schema
# ---------------------------------------------------------------------

SPEAKER_COLUMN = "speaker"
CONDITION_COLUMN = "recording_condition"
SCORE_COLUMN = "combined_score"
START_COLUMN = "samples_begin"

# Speaker-by-condition scores are represented by the median segment score.
SPEAKER_CONDITION_AGGREGATION = "median"

VALID_CONDITIONS = ("standard", "FG", "dialect")

CONDITION_MAPPING = {
    "standard": "standard",
    "wss": "standard",
    "ws_standard": "standard",
    "dialect": "dialect",
    "dialekt": "dialect",
    "wsd": "dialect",
    "ws_dialekt": "dialect",
    "fg": "FG",
}

# Display formatting
OMIT_LEADING_ZERO = True


## 3. Metric definitions

The compact table reports the following metrics:

- **Overall correlation:** Pearson correlation between speaker-by-condition DIME
  scores and the corresponding external D-values.
- **Separation (AUC):** segment-level discrimination between WSS and WSD, expressed
  as the area under the receiver operating characteristic curve.
- **Local trajectory variability:** weighted median of the within-recording standard
  deviations of segment-level DIME scores.
- **SEM:** median standard error of segment-level scores within speaker-by-condition
  recordings.
- **Dialect–FG contrast:** estimated WSD minus FG fixed-effect contrast from a mixed
  model with FG as the reference condition.
- **FG–Standard contrast:** estimated FG minus WSS fixed-effect contrast from the same
  mixed model.
- **Speaker variance:** variance of the speaker-specific random intercepts.
- **Residual variance:** residual segment-level variance not explained by condition
  or speaker-specific random intercepts.
- **Pooled lag-1 Pearson correlation:** Pearson correlation between each segment score
  and the immediately following score. Adjacent pairs are constructed within each
  speaker-by-condition recording and then pooled across recordings. Higher values
  indicate stronger persistence of absolute DIME-score levels.
- **Median adjacent absolute difference:** median absolute difference between consecutive
  scores within speaker-by-condition recordings. Lower values indicate greater
  numerical similarity between adjacent scores. Unlike the correlation, this metric
  directly penalizes large score jumps.
- **Baseline agreement:** Pearson correlation between speaker-by-condition aggregate
  scores from a given configuration and the corresponding scores from the baseline
  configuration.
- **Median absolute difference from baseline:** median absolute difference between
  matching speaker-by-condition aggregate scores from a given configuration and the
  baseline. Lower values indicate closer numerical agreement with the baseline.

Segments are ordered by `samples_begin` before adjacent pairs are formed. Rows with
missing speaker, condition, score, or segment-start information are excluded.


## 4. Input preparation

Condition labels are mapped to the common categories `standard`, `FG`, and `dialect`.
Speaker identifiers are stripped of surrounding whitespace and converted to uppercase.
Score and segment-start columns are converted to numeric values before invalid rows are
removed.


In [3]:
def read_table(path: Path) -> pd.DataFrame:
    """Read a CSV, Excel, Parquet, or pickle table."""
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix in {".pkl", ".pickle"}:
        return pd.read_pickle(path)

    raise ValueError(
        f"Unsupported table format {suffix!r} for {path}. "
        "Expected CSV, Excel, Parquet, or pickle."
    )


def normalize_condition(value: object) -> Optional[str]:
    """Map a condition label to standard, FG, or dialect."""
    if pd.isna(value):
        return None

    normalized = str(value).strip()
    mapped = CONDITION_MAPPING.get(normalized.lower())

    if mapped is not None:
        return mapped

    return normalized if normalized in VALID_CONDITIONS else None


def first_existing_column(
    frame: pd.DataFrame,
    candidates: Sequence[str],
    description: str,
) -> str:
    """Return the first available column from a list of candidates."""
    for column in candidates:
        if column in frame.columns:
            return column

    raise ValueError(
        f"Could not find a column for {description}. "
        f"Expected one of {list(candidates)}."
    )


def prepare_d_values(raw_d_values: pd.DataFrame) -> pd.DataFrame:
    """Standardize and aggregate the external D-value table."""
    frame = raw_d_values.copy()

    speaker_source = first_existing_column(
        frame,
        ["speaker", "sprecher"],
        "speaker IDs",
    )
    condition_source = first_existing_column(
        frame,
        ["recording_condition", "condition", "situation"],
        "recording conditions",
    )
    value_source = first_existing_column(
        frame,
        ["d_value", "d_value_mean", "value"],
        "D-values",
    )

    frame = frame.rename(
        columns={
            speaker_source: "speaker",
            condition_source: "condition",
            value_source: "d_value",
        }
    )

    frame["speaker"] = (
        frame["speaker"]
        .astype(str)
        .str.strip()
        .str.upper()
    )
    frame["condition"] = frame["condition"].map(
        normalize_condition
    )
    frame["d_value"] = pd.to_numeric(
        frame["d_value"],
        errors="coerce",
    )

    frame = (
        frame
        .replace([np.inf, -np.inf], np.nan)
        .dropna(
            subset=[
                "speaker",
                "condition",
                "d_value",
            ]
        )
        .loc[
            lambda x: x["condition"].isin(
                VALID_CONDITIONS
            )
        ]
        .copy()
    )

    # Repeated reference values are averaged within speaker and condition.
    return (
        frame
        .groupby(
            ["speaker", "condition"],
            as_index=False,
        )
        .agg(d_value=("d_value", "mean"))
    )


def prepare_segment_table(
    raw_segments: pd.DataFrame,
) -> pd.DataFrame:
    """Validate and standardize one segment-level DIME score table."""
    required_columns = {
        SPEAKER_COLUMN,
        CONDITION_COLUMN,
        SCORE_COLUMN,
        START_COLUMN,
    }
    missing_columns = required_columns.difference(
        raw_segments.columns
    )

    if missing_columns:
        raise ValueError(
            f"Missing required segment columns: "
            f"{sorted(missing_columns)}"
        )

    frame = raw_segments.copy()

    frame["speaker"] = (
        frame[SPEAKER_COLUMN]
        .astype(str)
        .str.strip()
        .str.upper()
    )
    frame["condition"] = frame[CONDITION_COLUMN].map(
        normalize_condition
    )
    frame["score"] = pd.to_numeric(
        frame[SCORE_COLUMN],
        errors="coerce",
    )
    frame["segment_start"] = pd.to_numeric(
        frame[START_COLUMN],
        errors="coerce",
    )

    frame = (
        frame
        .replace([np.inf, -np.inf], np.nan)
        .dropna(
            subset=[
                "speaker",
                "condition",
                "score",
                "segment_start",
            ]
        )
        .loc[
            lambda x: x["condition"].isin(
                VALID_CONDITIONS
            )
        ]
        .copy()
    )

    return frame


## 5. Statistical helper functions

In [4]:
def safe_correlation(
    x: Iterable[float],
    y: Iterable[float],
) -> Dict[str, float]:
    """Calculate Pearson and Spearman correlations after finite-value filtering."""
    x_array = np.asarray(list(x), dtype=float)
    y_array = np.asarray(list(y), dtype=float)

    valid = np.isfinite(x_array) & np.isfinite(y_array)
    x_array = x_array[valid]
    y_array = y_array[valid]

    result = {
        "n": int(len(x_array)),
        "pearson_r": np.nan,
        "pearson_p": np.nan,
        "spearman_rho": np.nan,
        "spearman_p": np.nan,
    }

    if (
        len(x_array) < 3
        or np.std(x_array) == 0
        or np.std(y_array) == 0
    ):
        return result

    pearson_r, pearson_p = pearsonr(x_array, y_array)
    spearman_rho, spearman_p = spearmanr(x_array, y_array)

    result.update(
        {
            "pearson_r": float(pearson_r),
            "pearson_p": float(pearson_p),
            "spearman_rho": float(spearman_rho),
            "spearman_p": float(spearman_p),
        }
    )
    return result


def weighted_median(
    values: Iterable[float],
    weights: Iterable[float],
) -> float:
    """Calculate a weighted median after removing invalid observations."""
    value_array = np.asarray(list(values), dtype=float)
    weight_array = np.asarray(list(weights), dtype=float)

    valid = (
        np.isfinite(value_array)
        & np.isfinite(weight_array)
        & (weight_array >= 0)
    )
    value_array = value_array[valid]
    weight_array = weight_array[valid]

    if len(value_array) == 0 or weight_array.sum() == 0:
        return np.nan

    order = np.argsort(value_array)
    sorted_values = value_array[order]
    sorted_weights = weight_array[order]

    cutoff = 0.5 * sorted_weights.sum()
    cumulative_weight = np.cumsum(sorted_weights)

    return float(
        sorted_values[
            np.searchsorted(cumulative_weight, cutoff)
        ]
    )


def cohens_d(
    group_a: Iterable[float],
    group_b: Iterable[float],
) -> float:
    """Calculate Cohen's d using the pooled sample standard deviation."""
    a = np.asarray(list(group_a), dtype=float)
    b = np.asarray(list(group_b), dtype=float)

    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]

    if len(a) < 2 or len(b) < 2:
        return np.nan

    pooled_variance = (
        (len(a) - 1) * np.var(a, ddof=1)
        + (len(b) - 1) * np.var(b, ddof=1)
    ) / (len(a) + len(b) - 2)

    if pooled_variance <= 0:
        return np.nan

    return float(
        (np.mean(a) - np.mean(b))
        / np.sqrt(pooled_variance)
    )


def cliffs_delta(
    group_a: Iterable[float],
    group_b: Iterable[float],
) -> float:
    """Calculate Cliff's delta efficiently via the Mann–Whitney U statistic."""
    a = np.asarray(list(group_a), dtype=float)
    b = np.asarray(list(group_b), dtype=float)

    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]

    if len(a) == 0 or len(b) == 0:
        return np.nan

    u_statistic = mannwhitneyu(
        a,
        b,
        alternative="two-sided",
        method="asymptotic",
    ).statistic

    return float(
        2.0 * u_statistic / (len(a) * len(b)) - 1.0
    )

def significance_stars(p_value: float) -> str:
    """Return conventional significance stars."""
    if not np.isfinite(p_value):
        return ""
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return ""


## 6. Diagnostics for one segment configuration

The mixed model is fitted at segment level with a random speaker intercept:

`score ~ recording condition + (1 | speaker)`

FG is the reference condition. The reported fixed-effect contrasts are WSD minus FG
and FG minus WSS.


In [5]:
def aggregate_speaker_conditions(
    segments: pd.DataFrame,
) -> pd.DataFrame:
    """Aggregate segment scores to one row per speaker and condition."""
    if SPEAKER_CONDITION_AGGREGATION not in {"mean", "median"}:
        raise ValueError(
            "SPEAKER_CONDITION_AGGREGATION must be 'mean' or 'median'."
        )

    aggregated = (
        segments
        .groupby(["speaker", "condition"], as_index=False)
        .agg(
            score_aggregated=(
                "score",
                SPEAKER_CONDITION_AGGREGATION,
            ),
            score_mean=("score", "mean"),
            score_standard_deviation=("score", "std"),
            score_variance=("score", "var"),
            segment_count=("score", "size"),
        )
    )

    aggregated["score_sem"] = (
        aggregated["score_standard_deviation"]
        / np.sqrt(aggregated["segment_count"])
    )

    single_segment = aggregated["segment_count"] < 2
    aggregated.loc[
        single_segment,
        [
            "score_standard_deviation",
            "score_variance",
            "score_sem",
        ],
    ] = np.nan

    return aggregated


def calculate_separation_metrics(
    segments: pd.DataFrame,
) -> Dict[str, float]:
    """Evaluate segment-level WSS versus WSD separation."""
    standard_scores = (
        segments.loc[
            segments["condition"] == "standard",
            "score",
        ]
        .dropna()
        .to_numpy(dtype=float)
    )
    dialect_scores = (
        segments.loc[
            segments["condition"] == "dialect",
            "score",
        ]
        .dropna()
        .to_numpy(dtype=float)
    )

    result = {
        "n_standard_segments": int(len(standard_scores)),
        "n_dialect_segments": int(len(dialect_scores)),
        "separation_auc": np.nan,
        "cohens_d_dialect_minus_standard": np.nan,
        "cliffs_delta_dialect_vs_standard": np.nan,
    }

    if len(standard_scores) > 0 and len(dialect_scores) > 0:
        labels = np.concatenate(
            [
                np.zeros(len(standard_scores)),
                np.ones(len(dialect_scores)),
            ]
        )
        scores = np.concatenate(
            [standard_scores, dialect_scores]
        )

        result["separation_auc"] = float(
            roc_auc_score(labels, scores)
        )
        result["cohens_d_dialect_minus_standard"] = cohens_d(
            dialect_scores,
            standard_scores,
        )
        result["cliffs_delta_dialect_vs_standard"] = cliffs_delta(
            dialect_scores,
            standard_scores,
        )

    return result

def calculate_temporal_diagnostics(
    segments: pd.DataFrame,
) -> Dict[str, float]:
    """
    Evaluate numerical consistency between consecutive segment scores.

    Adjacent pairs are constructed within each speaker-by-condition
    recording. All valid pairs are then pooled for one lag-1 Pearson
    correlation. No within-recording centering is applied.
    """
    result = {
        "n_adjacent_pairs": 0,
        "mean_adjacent_absolute_difference": np.nan,
        "median_adjacent_absolute_difference": np.nan,
        "standard_deviation_adjacent_absolute_difference": np.nan,
        "adjacent_pearson_r": np.nan,
        "adjacent_pearson_p": np.nan,
        "adjacent_spearman_rho": np.nan,
        "adjacent_spearman_p": np.nan,
    }

    grouping_columns = [
        "speaker",
        "condition",
    ]

    ordered = (
        segments
        .dropna(
            subset=[
                "speaker",
                "condition",
                "segment_start",
                "score",
            ]
        )
        .sort_values(
            grouping_columns + ["segment_start"]
        )
        .copy()
    )

    ordered["next_score"] = (
        ordered
        .groupby(grouping_columns)["score"]
        .shift(-1)
    )

    pairs = ordered.dropna(
        subset=[
            "score",
            "next_score",
        ]
    ).copy()

    if pairs.empty:
        return result

    pairs["adjacent_absolute_difference"] = np.abs(
        pairs["next_score"] - pairs["score"]
    )

    pooled_correlation = safe_correlation(
        pairs["score"],
        pairs["next_score"],
    )

    differences = pairs[
        "adjacent_absolute_difference"
    ]

    result.update(
        {
            "n_adjacent_pairs": int(len(pairs)),
            "mean_adjacent_absolute_difference": float(
                differences.mean()
            ),
            "median_adjacent_absolute_difference": float(
                differences.median()
            ),
            "standard_deviation_adjacent_absolute_difference": (
                float(differences.std(ddof=1))
                if len(differences) >= 2
                else np.nan
            ),
            "adjacent_pearson_r": pooled_correlation[
                "pearson_r"
            ],
            "adjacent_pearson_p": pooled_correlation[
                "pearson_p"
            ],
            "adjacent_spearman_rho": pooled_correlation[
                "spearman_rho"
            ],
            "adjacent_spearman_p": pooled_correlation[
                "spearman_p"
            ],
        }
    )

    return result

def calculate_ordering_diagnostics(
    aggregated: pd.DataFrame,
) -> Dict[str, float]:
    """Evaluate the expected within-speaker ordering WSD > FG > WSS."""
    wide = aggregated.pivot(
        index="speaker",
        columns="condition",
        values="score_aggregated",
    )

    required_conditions = ["dialect", "FG", "standard"]
    complete = wide.dropna(
        subset=required_conditions
    ).copy()

    result = {
        "n_complete_speakers": int(len(complete)),
        "proportion_dialect_above_fg": np.nan,
        "proportion_fg_above_standard": np.nan,
        "proportion_dialect_above_standard": np.nan,
        "proportion_full_ordering": np.nan,
        "mean_dialect_minus_fg": np.nan,
        "mean_fg_minus_standard": np.nan,
        "mean_dialect_minus_standard": np.nan,
        "median_dialect_minus_fg": np.nan,
        "median_fg_minus_standard": np.nan,
        "median_dialect_minus_standard": np.nan,
    }

    if complete.empty:
        return result

    dialect_minus_fg = (
        complete["dialect"]
        - complete["FG"]
    )
    fg_minus_standard = (
        complete["FG"]
        - complete["standard"]
    )
    dialect_minus_standard = (
        complete["dialect"]
        - complete["standard"]
    )

    result.update(
        {
            "proportion_dialect_above_fg": float(
                (dialect_minus_fg > 0).mean()
            ),
            "proportion_fg_above_standard": float(
                (fg_minus_standard > 0).mean()
            ),
            "proportion_dialect_above_standard": float(
                (dialect_minus_standard > 0).mean()
            ),
            "proportion_full_ordering": float(
                (
                    (dialect_minus_fg > 0)
                    & (fg_minus_standard > 0)
                ).mean()
            ),
            "mean_dialect_minus_fg": float(
                dialect_minus_fg.mean()
            ),
            "mean_fg_minus_standard": float(
                fg_minus_standard.mean()
            ),
            "mean_dialect_minus_standard": float(
                dialect_minus_standard.mean()
            ),
            "median_dialect_minus_fg": float(
                dialect_minus_fg.median()
            ),
            "median_fg_minus_standard": float(
                fg_minus_standard.median()
            ),
            "median_dialect_minus_standard": float(
                dialect_minus_standard.median()
            ),
        }
    )

    return result


def fit_segment_level_mixed_model(
    segments: pd.DataFrame,
) -> Dict[str, float]:
    """Fit a random-intercept model and return contrasts and variance components."""
    result = {
        "mixed_model_converged": np.nan,
        "dialect_minus_fg": np.nan,
        "dialect_minus_fg_standard_error": np.nan,
        "dialect_minus_fg_p": np.nan,
        "fg_minus_standard": np.nan,
        "fg_minus_standard_standard_error": np.nan,
        "fg_minus_standard_p": np.nan,
        "speaker_variance": np.nan,
        "residual_variance": np.nan,
        "residual_to_speaker_variance_ratio": np.nan,
    }

    model_data = (
        segments[
            ["speaker", "condition", "score"]
        ]
        .dropna()
        .copy()
    )

    if (
        model_data["speaker"].nunique() < 2
        or model_data["condition"].nunique() < 2
    ):
        return result

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            fitted_model = smf.mixedlm(
                formula=(
                    "score ~ "
                    "C(condition, Treatment(reference='FG'))"
                ),
                data=model_data,
                groups=model_data["speaker"],
                re_formula="1",
            ).fit(reml=True)

        dialect_key = (
            "C(condition, Treatment(reference='FG'))"
            "[T.dialect]"
        )
        standard_key = (
            "C(condition, Treatment(reference='FG'))"
            "[T.standard]"
        )

        dialect_minus_fg = fitted_model.params.get(
            dialect_key,
            np.nan,
        )
        standard_minus_fg = fitted_model.params.get(
            standard_key,
            np.nan,
        )

        dialect_se = fitted_model.bse.get(
            dialect_key,
            np.nan,
        )
        standard_se = fitted_model.bse.get(
            standard_key,
            np.nan,
        )

        dialect_p = fitted_model.pvalues.get(
            dialect_key,
            np.nan,
        )
        standard_p = fitted_model.pvalues.get(
            standard_key,
            np.nan,
        )

        speaker_variance = float(
            fitted_model.cov_re.iloc[0, 0]
        )
        residual_variance = float(
            fitted_model.scale
        )

        result.update(
            {
                "mixed_model_converged": bool(
                    fitted_model.converged
                ),
                "dialect_minus_fg": float(
                    dialect_minus_fg
                ),
                "dialect_minus_fg_standard_error": float(
                    dialect_se
                ),
                "dialect_minus_fg_p": float(
                    dialect_p
                ),
                "fg_minus_standard": float(
                    -standard_minus_fg
                ),
                "fg_minus_standard_standard_error": float(
                    standard_se
                ),
                "fg_minus_standard_p": float(
                    standard_p
                ),
                "speaker_variance": speaker_variance,
                "residual_variance": residual_variance,
                "residual_to_speaker_variance_ratio": (
                    residual_variance / speaker_variance
                    if speaker_variance > 0
                    else np.nan
                ),
            }
        )

    except Exception as error:
        result["mixed_model_error"] = str(error)

    return result


In [6]:
def evaluate_segment_configuration(
    label: str,
    path: Path,
    d_values: pd.DataFrame,
) -> Tuple[Dict[str, float], pd.DataFrame, pd.DataFrame]:
    """Calculate all diagnostics for one segment configuration."""
    raw_segments = read_table(path)
    segments = prepare_segment_table(raw_segments)
    aggregated = aggregate_speaker_conditions(segments)

    correlation_data = aggregated.merge(
        d_values,
        on=["speaker", "condition"],
        how="inner",
        validate="one_to_one",
    )

    overall_correlation = safe_correlation(
        correlation_data["score_aggregated"],
        correlation_data["d_value"],
    )

    separation = calculate_separation_metrics(segments)
    temporal = calculate_temporal_diagnostics(segments)
    ordering = calculate_ordering_diagnostics(aggregated)

    mixed_model = fit_segment_level_mixed_model(segments)

    aggregated_cell_variance = float(
        correlation_data["score_aggregated"].var(ddof=1)
    )
    aggregated_cell_standard_deviation = float(
        correlation_data["score_aggregated"].std(ddof=1)
    )

    median_within_variance = float(
        correlation_data["score_variance"].median()
    )
    weighted_median_within_standard_deviation = weighted_median(
        correlation_data["score_standard_deviation"],
        correlation_data["segment_count"],
    )

    result = {
        "configuration": label,
        "source_path": str(path),
        "n_raw_segment_rows": int(len(raw_segments)),
        "n_invalid_segment_rows_removed": int(
            len(raw_segments) - len(segments)
        ),
        "n_speakers": int(
            correlation_data["speaker"].nunique()
        ),
        "n_speaker_condition_cells": int(
            len(correlation_data)
        ),
        "n_segments_total": int(len(segments)),
        "mean_segments_per_cell": float(
            correlation_data["segment_count"].mean()
        ),
        "median_segments_per_cell": float(
            correlation_data["segment_count"].median()
        ),
        "correlation_n": overall_correlation["n"],
        "overall_pearson_r": overall_correlation["pearson_r"],
        "overall_pearson_p": overall_correlation["pearson_p"],
        "overall_spearman_rho": overall_correlation["spearman_rho"],
        "overall_spearman_p": overall_correlation["spearman_p"],
        "mean_within_variance": float(
            correlation_data["score_variance"].mean()
        ),
        "median_within_variance": median_within_variance,
        "mean_within_standard_deviation": float(
            correlation_data[
                "score_standard_deviation"
            ].mean()
        ),
        "median_within_standard_deviation": float(
            correlation_data[
                "score_standard_deviation"
            ].median()
        ),
        "weighted_median_within_standard_deviation": (
            weighted_median_within_standard_deviation
        ),
        "mean_sem": float(
            correlation_data["score_sem"].mean()
        ),
        "median_sem": float(
            correlation_data["score_sem"].median()
        ),
        "aggregated_cell_variance": (
            aggregated_cell_variance
        ),
        "aggregated_cell_standard_deviation": (
            aggregated_cell_standard_deviation
        ),
        "within_to_aggregated_variance_ratio": (
            median_within_variance
            / aggregated_cell_variance
            if aggregated_cell_variance > 0
            else np.nan
        ),
        "within_to_aggregated_sd_ratio": (
            weighted_median_within_standard_deviation
            / aggregated_cell_standard_deviation
            if aggregated_cell_standard_deviation > 0
            else np.nan
        ),
    }

    result.update(separation)
    result.update(temporal)
    result.update(ordering)
    result.update(mixed_model)

    return result, aggregated, segments


## 7. Evaluate all configured segment lengths

Each configuration is evaluated independently. Missing or invalid files are recorded
in `evaluation_errors` so that the remaining configurations can still be processed.


In [ ]:
raw_d_values = read_table(D_VALUES_PATH)
d_values = prepare_d_values(raw_d_values)

evaluation_rows = []
aggregated_scores_by_configuration = {}
prepared_segments_by_configuration = {}
evaluation_errors = []

for configuration, path in SEGMENT_SCORE_PATHS.items():
    start_time = perf_counter()
    print(f"Evaluating {configuration} ...", flush=True)

    try:
        (
            metrics, aggregated_scores, prepared_segments,
        ) = evaluate_segment_configuration(
            label=configuration,
            path=path,
            d_values=d_values,
        )
        evaluation_rows.append(metrics)
        aggregated_scores_by_configuration[
            configuration
        ] = aggregated_scores
        prepared_segments_by_configuration[
            configuration
        ] = prepared_segments

        elapsed_seconds = perf_counter() - start_time
        print(
            f"Finished {configuration} in "
            f"{elapsed_seconds:.1f} seconds.",
            flush=True,
        )

    except Exception as error:
        elapsed_seconds = perf_counter() - start_time
        evaluation_errors.append(
            {
                "configuration": configuration,
                "path": str(path),
                "error": str(error),
            }
        )
        print(
            f"Failed {configuration} after "
            f"{elapsed_seconds:.1f} seconds: {error}",
            flush=True,
        )

evaluation_results = pd.DataFrame(evaluation_rows)

if evaluation_results.empty:
    raise RuntimeError(
        "No segment configuration could be evaluated. "
        "Inspect evaluation_errors and the configured paths."
    )

evaluation_results = (
    evaluation_results
    .set_index("configuration")
    .reindex(
        [
            label
            for label in SEGMENT_SCORE_PATHS
            if label in evaluation_results["configuration"].values
        ]
    )
    .reset_index()
)

evaluation_errors = pd.DataFrame(evaluation_errors)

print(
    f"Successfully evaluated "
    f"{len(evaluation_results)} configurations."
)

if not evaluation_errors.empty:
    print("\nConfigurations that could not be evaluated:")
    display(evaluation_errors)


Evaluating 10s ...
Finished 10s in 0.8 seconds.
Evaluating 9s ...
Finished 9s in 0.8 seconds.
Evaluating 8s ...
Finished 8s in 0.9 seconds.
Evaluating 8s (4s ov.) ...
Finished 8s (4s ov.) in 2.6 seconds.
Evaluating 5s ...
Finished 5s in 2.1 seconds.
Evaluating 3s ...
Finished 3s in 5.9 seconds.
Evaluating 1s ...
Finished 1s in 8.1 seconds.
Successfully evaluated 7 configurations.


## 8. Corpus and analysis-sample statistics

The following tables distinguish several related sample sizes:

- **Segment data:** all valid segment rows after schema and missing-value filtering.
- **Analysis speakers/cells:** speakers and speaker-by-condition cells represented in
  the segment data.
- **Correlation speakers/cells:** the subset that can be matched to an external
  D-value. The correlation sample size is therefore a count of speaker-by-condition
  cells, not only a count of speakers.
- **Summed segment seconds:** window duration multiplied by the number of segments;
  overlapping audio is counted repeatedly.
- **Unique analyzed seconds:** union of the segment intervals within each
  speaker-by-condition recording; overlap is counted only once.

Duration is derived from `samples_begin`, `SAMPLE_RATE_HZ`, and
`WINDOW_LENGTH_SECONDS`. Verify the sampling rate and window-duration mapping in the
configuration cell before interpreting the duration statistics.


In [8]:
def interval_union_length_samples(
    starts: Iterable[float],
    interval_length_samples: int,
) -> float:
    """Return the union length of equal-length intervals in samples."""
    start_array = np.asarray(list(starts), dtype=float)
    start_array = start_array[np.isfinite(start_array)]

    if len(start_array) == 0:
        return 0.0
    if interval_length_samples <= 0:
        raise ValueError("interval_length_samples must be positive.")

    start_array = np.sort(start_array)
    current_start = float(start_array[0])
    current_end = current_start + interval_length_samples
    union_length = 0.0

    for start in start_array[1:]:
        start = float(start)
        end = start + interval_length_samples

        if start <= current_end:
            current_end = max(current_end, end)
        else:
            union_length += current_end - current_start
            current_start = start
            current_end = end

    union_length += current_end - current_start
    return float(union_length)


def count_complete_speakers(cells: pd.DataFrame) -> int:
    """Count speakers represented in every configured condition."""
    if cells.empty:
        return 0

    condition_counts = (
        cells[["speaker", "condition"]]
        .drop_duplicates()
        .groupby("speaker")["condition"]
        .nunique()
    )
    return int((condition_counts == len(VALID_CONDITIONS)).sum())


def calculate_corpus_statistics(
    prepared_segments: Mapping[str, pd.DataFrame],
    d_value_table: pd.DataFrame,
    results: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Build overview, condition-level, and D-value inventory tables."""
    if SAMPLE_RATE_HZ <= 0:
        raise ValueError("SAMPLE_RATE_HZ must be positive.")

    result_lookup = results.set_index("configuration")
    overview_rows = []
    condition_rows = []

    d_value_keys = (
        d_value_table[["speaker", "condition"]]
        .drop_duplicates()
        .assign(has_d_value=True)
    )

    for configuration in SEGMENT_SCORE_PATHS:
        if configuration not in prepared_segments:
            continue
        if configuration not in WINDOW_LENGTH_SECONDS:
            raise KeyError(
                f"Missing window duration for {configuration!r} in "
                "WINDOW_LENGTH_SECONDS."
            )

        segments = prepared_segments[configuration].copy()
        window_seconds = float(
            WINDOW_LENGTH_SECONDS[configuration]
        )
        window_samples = int(round(window_seconds * SAMPLE_RATE_HZ))

        cell_rows = []
        for (speaker, condition), group in segments.groupby(
            ["speaker", "condition"],
            sort=False,
        ):
            starts = group["segment_start"].to_numpy(dtype=float)
            unique_samples = interval_union_length_samples(
                starts,
                window_samples,
            )
            span_samples = (
                float(np.max(starts) - np.min(starts) + window_samples)
                if len(starts)
                else 0.0
            )

            cell_rows.append(
                {
                    "speaker": speaker,
                    "condition": condition,
                    "segment_count": int(len(group)),
                    "summed_segment_seconds": float(
                        len(group) * window_seconds
                    ),
                    "unique_analyzed_seconds": float(
                        unique_samples / SAMPLE_RATE_HZ
                    ),
                    "first_to_last_segment_span_seconds": float(
                        span_samples / SAMPLE_RATE_HZ
                    ),
                }
            )

        cells = pd.DataFrame(cell_rows).merge(
            d_value_keys,
            on=["speaker", "condition"],
            how="left",
            validate="one_to_one",
        )
        cells["has_d_value"] = cells["has_d_value"].eq(True)
        matched_cells = cells.loc[cells["has_d_value"]].copy()

        unique_seconds = float(cells["unique_analyzed_seconds"].sum())
        summed_seconds = float(cells["summed_segment_seconds"].sum())
        matched_unique_seconds = float(
            matched_cells["unique_analyzed_seconds"].sum()
        )

        raw_rows = (
            int(result_lookup.loc[configuration, "n_raw_segment_rows"])
            if configuration in result_lookup.index
            else np.nan
        )
        invalid_rows = (
            int(
                result_lookup.loc[
                    configuration,
                    "n_invalid_segment_rows_removed",
                ]
            )
            if configuration in result_lookup.index
            else np.nan
        )

        overview_rows.append(
            {
                "configuration": configuration,
                "window_seconds": window_seconds,
                "raw_segment_rows": raw_rows,
                "valid_segments": int(len(segments)),
                "invalid_rows_removed": invalid_rows,
                "invalid_rows_percent": (
                    100.0 * invalid_rows / raw_rows
                    if raw_rows and np.isfinite(raw_rows)
                    else np.nan
                ),
                "summed_segment_seconds": summed_seconds,
                "unique_analyzed_seconds": unique_seconds,
                "unique_analyzed_hours": unique_seconds / 3600.0,
                "segment_redundancy_factor": (
                    summed_seconds / unique_seconds
                    if unique_seconds > 0
                    else np.nan
                ),
                "analysis_speakers": int(cells["speaker"].nunique()),
                "analysis_speaker_condition_cells": int(len(cells)),
                "complete_analysis_speakers": count_complete_speakers(cells),
                "correlation_speakers": int(
                    matched_cells["speaker"].nunique()
                ),
                "correlation_speaker_condition_cells": int(
                    len(matched_cells)
                ),
                "complete_correlation_speakers": count_complete_speakers(
                    matched_cells
                ),
                "cells_without_d_value": int(
                    (~cells["has_d_value"]).sum()
                ),
                "d_value_cell_coverage_percent": (
                    100.0 * len(matched_cells) / len(cells)
                    if len(cells) > 0
                    else np.nan
                ),
                "segments_contributing_to_correlation": int(
                    matched_cells["segment_count"].sum()
                ),
                "unique_seconds_contributing_to_correlation": (
                    matched_unique_seconds
                ),
                "median_segments_per_cell": float(
                    cells["segment_count"].median()
                ),
                "median_unique_seconds_per_cell": float(
                    cells["unique_analyzed_seconds"].median()
                ),
            }
        )

        for condition in VALID_CONDITIONS:
            condition_cells = cells.loc[
                cells["condition"] == condition
            ].copy()
            condition_matched = condition_cells.loc[
                condition_cells["has_d_value"]
            ]

            condition_rows.append(
                {
                    "configuration": configuration,
                    "condition": condition,
                    "speakers": int(
                        condition_cells["speaker"].nunique()
                    ),
                    "speaker_condition_cells": int(len(condition_cells)),
                    "segments": int(
                        condition_cells["segment_count"].sum()
                    ),
                    "summed_segment_seconds": float(
                        condition_cells["summed_segment_seconds"].sum()
                    ),
                    "unique_analyzed_seconds": float(
                        condition_cells["unique_analyzed_seconds"].sum()
                    ),
                    "unique_analyzed_hours": float(
                        condition_cells["unique_analyzed_seconds"].sum()
                        / 3600.0
                    ),
                    "matched_d_value_cells": int(
                        len(condition_matched)
                    ),
                    "matched_d_value_speakers": int(
                        condition_matched["speaker"].nunique()
                    ),
                    "cells_without_d_value": int(
                        (~condition_cells["has_d_value"]).sum()
                    ),
                    "d_value_cell_coverage_percent": (
                        100.0
                        * len(condition_matched)
                        / len(condition_cells)
                        if len(condition_cells) > 0
                        else np.nan
                    ),
                    "segments_contributing_to_correlation": int(
                        condition_matched["segment_count"].sum()
                    ),
                    "unique_seconds_contributing_to_correlation": float(
                        condition_matched[
                            "unique_analyzed_seconds"
                        ].sum()
                    ),
                    "median_segments_per_speaker": float(
                        condition_cells["segment_count"].median()
                    ),
                    "median_unique_seconds_per_speaker": float(
                        condition_cells[
                            "unique_analyzed_seconds"
                        ].median()
                    ),
                }
            )

    overview = pd.DataFrame(overview_rows)
    by_condition = pd.DataFrame(condition_rows)

    d_value_rows = [
        {
            "condition": "all conditions",
            "d_value_speaker_condition_cells": int(len(d_value_table)),
            "d_value_speakers": int(d_value_table["speaker"].nunique()),
            "complete_d_value_speakers": count_complete_speakers(
                d_value_table
            ),
        }
    ]
    for condition in VALID_CONDITIONS:
        subset = d_value_table.loc[
            d_value_table["condition"] == condition
        ]
        d_value_rows.append(
            {
                "condition": condition,
                "d_value_speaker_condition_cells": int(len(subset)),
                "d_value_speakers": int(subset["speaker"].nunique()),
                "complete_d_value_speakers": np.nan,
            }
        )

    d_value_inventory = pd.DataFrame(d_value_rows)
    return overview, by_condition, d_value_inventory


def rounded_statistics_table(frame: pd.DataFrame) -> pd.DataFrame:
    """Round floating-point statistics without converting counts to strings."""
    rounded = frame.copy()
    float_columns = rounded.select_dtypes(include=["float"]).columns
    rounded[float_columns] = rounded[float_columns].round(3)
    return rounded


(
    dataset_statistics_overview,
    dataset_statistics_by_condition,
    d_value_inventory,
) = calculate_corpus_statistics(
    prepared_segments=prepared_segments_by_configuration,
    d_value_table=d_values,
    results=evaluation_results,
)

print("Dataset and analysis-sample overview")
display(rounded_statistics_table(dataset_statistics_overview))

print("\nStatistics by recording condition")
display(rounded_statistics_table(dataset_statistics_by_condition))

print("\nAvailable external D-values")
display(rounded_statistics_table(d_value_inventory))


Dataset and analysis-sample overview


,configuration,window_seconds,raw_segment_rows,valid_segments,invalid_rows_removed,invalid_rows_percent,summed_segment_seconds,unique_analyzed_seconds,unique_analyzed_hours,segment_redundancy_factor,...,complete_analysis_speakers,correlation_speakers,correlation_speaker_condition_cells,complete_correlation_speakers,cells_without_d_value,d_value_cell_coverage_percent,segments_contributing_to_correlation,unique_seconds_contributing_to_correlation,median_segments_per_cell,median_unique_seconds_per_cell
0,10s,10.0,25289,25289,0,0.0,252890.0,252890.0,70.247,1.000,...,450,149,429,133,1112,27.839,5547,55470.0,16.0,160.0
1,9s,9.0,28176,28176,0,0.0,253584.0,253584.0,70.440,1.000,...,450,149,429,133,1112,27.839,6185,55665.0,17.0,153.0
2,8s,8.0,31815,31815,0,0.0,254520.0,254520.0,70.700,1.000,...,451,149,429,133,1113,27.821,6985,55880.0,20.0,160.0
3,8s (4s ov.),8.0,62848,62848,0,0.0,502784.0,257560.0,71.544,1.952,...,451,149,429,133,1113,27.821,13758,56748.0,39.0,160.0
4,5s,5.0,51351,51351,0,0.0,256755.0,256755.0,71.321,1.000,...,453,149,429,133,1115,27.785,11296,56480.0,32.0,160.0
5,3s,3.0,86109,86109,0,0.0,258327.0,258327.0,71.758,1.000,...,454,149,429,133,1116,27.767,18980,56940.0,53.0,159.0
6,1s,1.0,259928,259928,0,0.0,259928.0,259928.0,72.202,1.000,...,462,149,429,133,1127,27.571,57375,57375.0,160.0,160.0



Statistics by recording condition


,configuration,condition,speakers,speaker_condition_cells,segments,summed_segment_seconds,unique_analyzed_seconds,unique_analyzed_hours,matched_d_value_cells,matched_d_value_speakers,cells_without_d_value,d_value_cell_coverage_percent,segments_contributing_to_correlation,unique_seconds_contributing_to_correlation,median_segments_per_speaker,median_unique_seconds_per_speaker
0,10s,standard,532,532,8372,83720.0,83720.0,23.256,147,147,385,27.632,2250,22500.0,16.0,160.0
1,10s,FG,486,486,8570,85700.0,85700.0,23.806,146,146,340,30.041,1134,11340.0,10.0,100.0
2,10s,dialect,523,523,8347,83470.0,83470.0,23.186,136,136,387,26.004,2163,21630.0,16.0,160.0
3,9s,standard,532,532,9333,83997.0,83997.0,23.332,147,147,385,27.632,2509,22581.0,18.0,162.0
4,9s,FG,486,486,9550,85950.0,85950.0,23.875,146,146,340,30.041,1278,11502.0,11.0,99.0
5,9s,dialect,523,523,9293,83637.0,83637.0,23.232,136,136,387,26.004,2398,21582.0,17.0,153.0
6,8s,standard,533,533,10537,84296.0,84296.0,23.416,147,147,386,27.580,2828,22624.0,20.0,160.0
7,8s,FG,486,486,10764,86112.0,86112.0,23.920,146,146,340,30.041,1437,11496.0,13.0,104.0
8,8s,dialect,523,523,10514,84112.0,84112.0,23.364,136,136,387,26.004,2720,21760.0,20.0,160.0
9,8s (4s ov.),standard,533,533,20801,166408.0,85336.0,23.704,147,147,386,27.580,5580,22908.0,40.0,164.0



Available external D-values


,condition,d_value_speaker_condition_cells,d_value_speakers,complete_d_value_speakers
0,all conditions,520,179,165.0
1,standard,177,177,NaN
2,FG,176,176,NaN
3,dialect,167,167,NaN


## 9. Agreement with the baseline configuration

Baseline comparisons use matching speaker-by-condition aggregate scores. Pearson
correlation quantifies relative agreement, whereas the median absolute difference
quantifies numerical distance from the baseline on the DIME scale.


In [9]:
def compare_with_baseline(
    candidate: pd.DataFrame,
    baseline: pd.DataFrame,
) -> Dict[str, float]:
    """Compare matching speaker-by-condition aggregate scores."""
    comparison = candidate.merge(
        baseline,
        on=["speaker", "condition"],
        how="inner",
        suffixes=("_candidate", "_baseline"),
        validate="one_to_one",
    )

    correlation = safe_correlation(
        comparison["score_aggregated_candidate"],
        comparison["score_aggregated_baseline"],
    )

    absolute_difference = np.abs(
        comparison["score_aggregated_candidate"]
        - comparison["score_aggregated_baseline"]
    )

    return {
        "baseline_overlap_n": int(len(comparison)),
        "baseline_pearson_r": correlation["pearson_r"],
        "baseline_pearson_p": correlation["pearson_p"],
        "baseline_spearman_rho": correlation[
            "spearman_rho"
        ],
        "baseline_spearman_p": correlation[
            "spearman_p"
        ],
        "baseline_mean_absolute_difference": float(
            absolute_difference.mean()
        ),
        "baseline_median_absolute_difference": float(
            absolute_difference.median()
        ),
    }


if (
    BASELINE_CONFIGURATION
    not in aggregated_scores_by_configuration
):
    raise ValueError(
        f"Baseline configuration "
        f"{BASELINE_CONFIGURATION!r} "
        "was not evaluated successfully."
    )

baseline_scores = aggregated_scores_by_configuration[
    BASELINE_CONFIGURATION
]

baseline_rows = []

for configuration in evaluation_results["configuration"]:
    if configuration == BASELINE_CONFIGURATION:
        baseline_rows.append(
            {
                "configuration": configuration,
                "baseline_overlap_n": len(
                    baseline_scores
                ),
                "baseline_pearson_r": 1.0,
                "baseline_pearson_p": 0.0,
                "baseline_spearman_rho": 1.0,
                "baseline_spearman_p": 0.0,
                "baseline_mean_absolute_difference": 0.0,
                "baseline_median_absolute_difference": 0.0,
            }
        )
        continue

    baseline_rows.append(
        {
            "configuration": configuration,
            **compare_with_baseline(
                candidate=(
                    aggregated_scores_by_configuration[
                        configuration
                    ]
                ),
                baseline=baseline_scores,
            ),
        }
    )

baseline_results = pd.DataFrame(baseline_rows)

evaluation_results = evaluation_results.merge(
    baseline_results,
    on="configuration",
    how="left",
    validate="one_to_one",
)


## 10. Format compact and comprehensive tables

Correlation coefficients are annotated with conventional significance stars:

- `*`: p < .05
- `**`: p < .01
- `***`: p < .001


In [10]:
def format_decimal(
    value: object,
    digits: int = 3,
) -> str:
    """Format a numeric table value."""
    try:
        numeric = float(value)
    except (TypeError, ValueError):
        return "—"

    if not np.isfinite(numeric):
        return "—"

    formatted = f"{numeric:.{digits}f}"

    if OMIT_LEADING_ZERO:
        if formatted.startswith("0."):
            formatted = formatted[1:]
        elif formatted.startswith("-0."):
            formatted = "-" + formatted[2:]

    return formatted


def format_integer(value: object) -> str:
    """Format a count without decimal places."""
    try:
        numeric = float(value)
    except (TypeError, ValueError):
        return "—"

    if not np.isfinite(numeric):
        return "—"

    return f"{int(round(numeric)):,}"


def format_p_value(value: object) -> str:
    """Format a p-value using fixed or scientific notation."""
    try:
        numeric = float(value)
    except (TypeError, ValueError):
        return "—"

    if not np.isfinite(numeric):
        return "—"

    if numeric < 0.001:
        return f"{numeric:.2e}"

    return format_decimal(numeric, digits=3)


def format_correlation(
    value: object,
    p_value: object,
) -> str:
    """Format a correlation coefficient with significance stars."""
    try:
        numeric = float(value)
        p_numeric = float(p_value)
    except (TypeError, ValueError):
        return "—"

    if not np.isfinite(numeric):
        return "—"

    return (
        format_decimal(numeric, digits=3)
        + significance_stars(p_numeric)
    )


MetricSpec = Tuple[str, str, str, Optional[str]]


COMPACT_METRICS: List[MetricSpec] = [
    (
        "Overall correlation",
        "overall_pearson_r",
        "correlation",
        "overall_pearson_p",
    ),
    (
        "Separation (AUC)",
        "separation_auc",
        "decimal",
        None,
    ),
    (
        "Local trajectory variability",
        "weighted_median_within_standard_deviation",
        "decimal",
        None,
    ),
    (
        "SEM",
        "median_sem",
        "decimal",
        None,
    ),
    (
        "Dialect–FG contrast",
        "dialect_minus_fg",
        "decimal",
        None,
    ),
    (
        "FG–Standard contrast",
        "fg_minus_standard",
        "decimal",
        None,
    ),
    (
        "Speaker variance",
        "speaker_variance",
        "decimal",
        None,
    ),
    (
        "Residual variance",
        "residual_variance",
        "decimal",
        None,
    ),
    (
        "Pooled lag-1 Pearson correlation",
        "adjacent_pearson_r",
        "correlation",
        "adjacent_pearson_p",
    ),
    (
        "Median adjacent absolute difference",
        "median_adjacent_absolute_difference",
        "decimal",
        None,
    ),
    (
        "Baseline agreement",
        "baseline_pearson_r",
        "correlation",
        "baseline_pearson_p",
    ),
    (
        "Median absolute difference from baseline",
        "baseline_median_absolute_difference",
        "decimal",
        None,
    ),
]


FULL_METRICS: List[MetricSpec] = [
    ("Speakers", "n_speakers", "integer", None),
    (
        "Speaker × condition cells",
        "n_speaker_condition_cells",
        "integer",
        None,
    ),
    ("Segments", "n_segments_total", "integer", None),
    (
        "Mean segments per cell",
        "mean_segments_per_cell",
        "decimal",
        None,
    ),
    (
        "Median segments per cell",
        "median_segments_per_cell",
        "decimal",
        None,
    ),
    ("Correlation N", "correlation_n", "integer", None),
    (
        "Overall Pearson r",
        "overall_pearson_r",
        "correlation",
        "overall_pearson_p",
    ),
    ("Overall Pearson p", "overall_pearson_p", "p", None),
    (
        "Overall Spearman rho",
        "overall_spearman_rho",
        "correlation",
        "overall_spearman_p",
    ),
    ("Overall Spearman p", "overall_spearman_p", "p", None),
    ("Separation AUC", "separation_auc", "decimal", None),
    (
        "Cohen's d: WSD − WSS",
        "cohens_d_dialect_minus_standard",
        "decimal",
        None,
    ),
    (
        "Cliff's delta: WSD vs. WSS",
        "cliffs_delta_dialect_vs_standard",
        "decimal",
        None,
    ),
    (
        "WSS segments",
        "n_standard_segments",
        "integer",
        None,
    ),
    (
        "WSD segments",
        "n_dialect_segments",
        "integer",
        None,
    ),
    (
        "Mean within-cell variance",
        "mean_within_variance",
        "decimal",
        None,
    ),
    (
        "Median within-cell variance",
        "median_within_variance",
        "decimal",
        None,
    ),
    (
        "Mean within-cell SD",
        "mean_within_standard_deviation",
        "decimal",
        None,
    ),
    (
        "Median within-cell SD",
        "median_within_standard_deviation",
        "decimal",
        None,
    ),
    (
        "Weighted median within-cell SD",
        "weighted_median_within_standard_deviation",
        "decimal",
        None,
    ),
    ("Mean SEM", "mean_sem", "decimal", None),
    ("Median SEM", "median_sem", "decimal", None),
    (
        "Aggregated-cell variance",
        "aggregated_cell_variance",
        "decimal",
        None,
    ),
    (
        "Aggregated-cell SD",
        "aggregated_cell_standard_deviation",
        "decimal",
        None,
    ),
    (
        "Within/aggregated variance ratio",
        "within_to_aggregated_variance_ratio",
        "decimal",
        None,
    ),
    (
        "Within/aggregated SD ratio",
        "within_to_aggregated_sd_ratio",
        "decimal",
        None,
    ),
    (
        "Dialect–FG contrast",
        "dialect_minus_fg",
        "decimal",
        None,
    ),
    (
        "Dialect–FG contrast SE",
        "dialect_minus_fg_standard_error",
        "decimal",
        None,
    ),
    (
        "Dialect–FG contrast p",
        "dialect_minus_fg_p",
        "p",
        None,
    ),
    (
        "FG–Standard contrast",
        "fg_minus_standard",
        "decimal",
        None,
    ),
    (
        "FG–Standard contrast SE",
        "fg_minus_standard_standard_error",
        "decimal",
        None,
    ),
    (
        "FG–Standard contrast p",
        "fg_minus_standard_p",
        "p",
        None,
    ),
    (
        "Speaker variance",
        "speaker_variance",
        "decimal",
        None,
    ),
    (
        "Residual variance",
        "residual_variance",
        "decimal",
        None,
    ),
    (
        "Residual/speaker variance ratio",
        "residual_to_speaker_variance_ratio",
        "decimal",
        None,
    ),
    (
        "Adjacent segment pairs",
        "n_adjacent_pairs",
        "integer",
        None,
    ),
    (
        "Mean adjacent absolute difference",
        "mean_adjacent_absolute_difference",
        "decimal",
        None,
    ),
    (
        "Median adjacent absolute difference",
        "median_adjacent_absolute_difference",
        "decimal",
        None,
    ),
    (
        "SD of adjacent absolute differences",
        "standard_deviation_adjacent_absolute_difference",
        "decimal",
        None,
    ),
    (
        "Pooled lag-1 Pearson r",
        "adjacent_pearson_r",
        "correlation",
        "adjacent_pearson_p",
    ),
    (
        "Pooled lag-1 Pearson p",
        "adjacent_pearson_p",
        "p",
        None,
    ),
    (
        "Adjacent Spearman rho",
        "adjacent_spearman_rho",
        "correlation",
        "adjacent_spearman_p",
    ),
    (
        "Adjacent Spearman p",
        "adjacent_spearman_p",
        "p",
        None,
    ),
    (
        "Complete speakers for ordering",
        "n_complete_speakers",
        "integer",
        None,
    ),
    (
        "Proportion WSD > FG",
        "proportion_dialect_above_fg",
        "decimal",
        None,
    ),
    (
        "Proportion FG > WSS",
        "proportion_fg_above_standard",
        "decimal",
        None,
    ),
    (
        "Proportion WSD > WSS",
        "proportion_dialect_above_standard",
        "decimal",
        None,
    ),
    (
        "Proportion WSD > FG > WSS",
        "proportion_full_ordering",
        "decimal",
        None,
    ),
    (
        "Mean WSD − FG",
        "mean_dialect_minus_fg",
        "decimal",
        None,
    ),
    (
        "Mean FG − WSS",
        "mean_fg_minus_standard",
        "decimal",
        None,
    ),
    (
        "Mean WSD − WSS",
        "mean_dialect_minus_standard",
        "decimal",
        None,
    ),
    (
        "Median WSD − FG",
        "median_dialect_minus_fg",
        "decimal",
        None,
    ),
    (
        "Median FG − WSS",
        "median_fg_minus_standard",
        "decimal",
        None,
    ),
    (
        "Median WSD − WSS",
        "median_dialect_minus_standard",
        "decimal",
        None,
    ),
    (
        "Baseline overlap N",
        "baseline_overlap_n",
        "integer",
        None,
    ),
    (
        "Baseline Pearson r",
        "baseline_pearson_r",
        "correlation",
        "baseline_pearson_p",
    ),
    (
        "Baseline Pearson p",
        "baseline_pearson_p",
        "p",
        None,
    ),
    (
        "Baseline Spearman rho",
        "baseline_spearman_rho",
        "correlation",
        "baseline_spearman_p",
    ),
    (
        "Baseline Spearman p",
        "baseline_spearman_p",
        "p",
        None,
    ),
    (
        "Mean absolute difference from baseline",
        "baseline_mean_absolute_difference",
        "decimal",
        None,
    ),
    (
        "Median absolute difference from baseline",
        "baseline_median_absolute_difference",
        "decimal",
        None,
    ),
]


def build_formatted_metric_table(
    results: pd.DataFrame,
    metric_specs: Sequence[MetricSpec],
) -> pd.DataFrame:
    """Build a metric-by-configuration table of formatted strings."""
    indexed = results.set_index("configuration")

    required_columns = {
        value_column
        for _, value_column, _, _ in metric_specs
    }
    required_columns.update(
        p_column
        for _, _, _, p_column in metric_specs
        if p_column is not None
    )

    missing_columns = required_columns.difference(
        indexed.columns
    )
    if missing_columns:
        raise KeyError(
            "The evaluation results do not contain all "
            "requested metric columns: "
            f"{sorted(missing_columns)}. "
            "Rerun the evaluation cells after changing "
            "a metric function."
        )

    configurations = [
        label
        for label in SEGMENT_SCORE_PATHS
        if label in indexed.index
    ]

    rows = []

    for display_name, value_column, format_type, p_column in metric_specs:
        row = {"Metric": display_name}

        for configuration in configurations:
            value = indexed.loc[configuration, value_column]

            if (
                configuration == BASELINE_CONFIGURATION
                and value_column.startswith("baseline_")
            ):
                row[configuration] = "—"
                continue

            if format_type == "integer":
                row[configuration] = format_integer(value)
            elif format_type == "p":
                row[configuration] = format_p_value(value)
            elif format_type == "correlation":
                p_value = (
                    indexed.loc[configuration, p_column]
                    if p_column is not None
                    else np.nan
                )
                row[configuration] = format_correlation(
                    value,
                    p_value,
                )
            else:
                row[configuration] = format_decimal(value)

        rows.append(row)

    return pd.DataFrame(rows).set_index("Metric")


compact_metric_table = build_formatted_metric_table(
    evaluation_results,
    COMPACT_METRICS,
)

full_metric_table = build_formatted_metric_table(
    evaluation_results,
    FULL_METRICS,
)

print("Compact evaluation table")
display(compact_metric_table)

print("\nComprehensive diagnostics table")
display(full_metric_table)


Compact evaluation table


,10s,9s,8s,8s (4s ov.),5s,3s,1s
Metric,,,,,,,
Overall correlation,.686***,.691***,.692***,.706***,.701***,.719***,.644***
Separation (AUC),.793,.802,.794,.804,.763,.749,.640
Local trajectory variability,.545,.565,.586,.593,.658,.700,.661
SEM,.154,.147,.141,.106,.126,.106,.058
Dialect–FG contrast,.386,.367,.361,.380,.319,.248,.098
FG–Standard contrast,.660,.697,.654,.693,.548,.530,.259
Speaker variance,.375,.348,.321,.327,.228,.156,.066
Residual variance,.432,.454,.459,.480,.526,.559,.469
Pooled lag-1 Pearson correlation,.683***,.663***,.629***,.763***,.495***,.396***,.214***



Comprehensive diagnostics table


,10s,9s,8s,8s (4s ov.),5s,3s,1s
Metric,,,,,,,
Speakers,149,149,149,149,149,149,149
Speaker × condition cells,429,429,429,429,429,429,429
Segments,"25,289","28,176","31,815","62,848","51,351","86,109","259,928"
Mean segments per cell,12.930,14.417,16.282,32.070,26.331,44.242,133.741
Median segments per cell,14.000,15.000,17.000,34.000,28.000,46.000,140.000
...,...,...,...,...,...,...,...
Baseline Pearson p,—,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,4.26e-267
Baseline Spearman rho,—,.974***,.979***,.978***,.969***,.926***,.753***
Baseline Spearman p,—,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,7.71e-282


## 11. Pairwise agreement between all configurations

The baseline comparison uses one designated reference configuration. The following
table additionally compares every available pair of window configurations.


In [11]:
def calculate_pairwise_agreement(
    aggregated_scores: Mapping[str, pd.DataFrame],
) -> pd.DataFrame:
    """Compare every pair of aggregated segment configurations."""
    configurations = [
        label
        for label in SEGMENT_SCORE_PATHS
        if label in aggregated_scores
    ]

    rows = []

    for index_a, configuration_a in enumerate(configurations):
        for configuration_b in configurations[index_a + 1:]:
            comparison = aggregated_scores[
                configuration_a
            ].merge(
                aggregated_scores[configuration_b],
                on=["speaker", "condition"],
                how="inner",
                suffixes=("_a", "_b"),
                validate="one_to_one",
            )

            correlation = safe_correlation(
                comparison["score_aggregated_a"],
                comparison["score_aggregated_b"],
            )

            absolute_difference = np.abs(
                comparison["score_aggregated_a"]
                - comparison["score_aggregated_b"]
            )

            rows.append(
                {
                    "configuration_a": configuration_a,
                    "configuration_b": configuration_b,
                    "n_overlap": len(comparison),
                    "pearson_r": correlation["pearson_r"],
                    "pearson_p": correlation["pearson_p"],
                    "spearman_rho": correlation["spearman_rho"],
                    "spearman_p": correlation["spearman_p"],
                    "mean_absolute_difference": float(
                        absolute_difference.mean()
                    ),
                    "median_absolute_difference": float(
                        absolute_difference.median()
                    ),
                }
            )

    return pd.DataFrame(rows)


pairwise_agreement = calculate_pairwise_agreement(
    aggregated_scores_by_configuration
)

display(pairwise_agreement)


,configuration_a,configuration_b,n_overlap,pearson_r,pearson_p,spearman_rho,spearman_p,mean_absolute_difference,median_absolute_difference
0,10s,9s,1541,0.972584,0.000000e+00,0.973954,0.000000e+00,0.151921,0.121575
1,10s,8s,1541,0.979323,0.000000e+00,0.978558,0.000000e+00,0.138653,0.114224
2,10s,8s (4s ov.),1541,0.979490,0.000000e+00,0.978084,0.000000e+00,0.145114,0.125423
3,10s,5s,1541,0.971352,0.000000e+00,0.969294,0.000000e+00,0.209192,0.170891
4,10s,3s,1541,0.927813,0.000000e+00,0.926084,0.000000e+00,0.320139,0.251528
5,10s,1s,1541,0.739757,4.258350e-267,0.752841,7.707763e-282,0.622935,0.482491
6,9s,8s,1541,0.979184,0.000000e+00,0.979521,0.000000e+00,0.130377,0.106457
7,9s,8s (4s ov.),1541,0.981691,0.000000e+00,0.982205,0.000000e+00,0.146073,0.123642
8,9s,5s,1541,0.963028,0.000000e+00,0.962863,0.000000e+00,0.208886,0.170502
9,9s,3s,1541,0.942538,0.000000e+00,0.941519,0.000000e+00,0.293712,0.240910
